# Torque Sensor Data Analysis

Load and visualize CSV data from the Torque Cell DAQ system (DYJN-104 + NI USB-6002).

In [ ]:
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import numpy as np

## 1. Load CSV Data

Select which CSV file to analyze. The loader skips the metadata comment lines automatically.

In [ ]:
# List available data files
data_dir = Path("../data")
csv_files = sorted(data_dir.glob("torque_log_*.csv"))

print(f"Found {len(csv_files)} data files:")
for i, f in enumerate(csv_files):
    size_kb = f.stat().st_size / 1024
    print(f"  [{i}] {f.name}  ({size_kb:.1f} KB)")

In [ ]:
# Select file index (change this to load a different file)
FILE_INDEX = -1  # -1 = most recent

csv_path = csv_files[FILE_INDEX]
print(f"Loading: {csv_path.name}")

# Read metadata from comment lines
metadata = {}
with open(csv_path) as f:
    for line in f:
        if not line.startswith("#"):
            break
        key, _, val = line[2:].strip().partition(": ")
        metadata[key] = val

print("\nMetadata:")
for k, v in metadata.items():
    print(f"  {k}: {v}")

# Load data
df = pd.read_csv(csv_path, comment="#")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Replace empty torque with NaN
if "torque_Nm" in df.columns:
    df["torque_Nm"] = pd.to_numeric(df["torque_Nm"], errors="coerce")

has_torque = df["torque_Nm"].notna().any() if "torque_Nm" in df.columns else False

print(f"\nLoaded {len(df)} samples")
print(f"Duration: {df['elapsed_s'].iloc[-1]:.2f} s")
print(f"Sample rate: {metadata.get('Sample Rate', 'unknown')} Hz")
print(f"Torque calibration: {'Yes' if has_torque else 'No (raw voltage only)'}")
df.head()

## 2. Voltage vs Time

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=df["elapsed_s"],
    y=df["voltage_V"],
    mode="lines",
    name="Voltage",
    line=dict(color="#89b4fa", width=1.2),
))
fig.update_layout(
    title=f"Voltage vs Time — {csv_path.name}",
    xaxis_title="Elapsed Time (s)",
    yaxis_title="Voltage (V)",
    template="plotly_dark",
    height=450,
)
fig.show()

## 3. Torque vs Time

Only plotted if calibration data was applied during recording. If not, you can apply a linear calibration below.

In [ ]:
# Apply post-hoc calibration if torque column is empty
# Set these values to match your DYJN-104 + 510 transmitter setup
APPLY_CALIBRATION = not has_torque  # Only apply if no calibration exists
CAL_SLOPE = 10.0    # Nm per Volt
CAL_OFFSET = 0.0    # Nm

if APPLY_CALIBRATION:
    df["torque_Nm"] = CAL_SLOPE * df["voltage_V"] + CAL_OFFSET
    has_torque = True
    print(f"Applied post-hoc calibration: torque = {CAL_SLOPE} * voltage + {CAL_OFFSET}")
else:
    print("Using calibration from recording session")

In [ ]:
if has_torque:
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=df["elapsed_s"],
        y=df["torque_Nm"],
        mode="lines",
        name="Torque",
        line=dict(color="#a6e3a1", width=1.2),
    ))
    fig.update_layout(
        title=f"Torque vs Time — {csv_path.name}",
        xaxis_title="Elapsed Time (s)",
        yaxis_title="Torque (Nm)",
        template="plotly_dark",
        height=450,
    )
    fig.show()
else:
    print("No torque data available. Set APPLY_CALIBRATION = True above.")

## 4. Combined Plot (Dual Y-Axis)

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scattergl(x=df["elapsed_s"], y=df["voltage_V"], name="Voltage (V)",
                 line=dict(color="#89b4fa", width=1.2)),
    secondary_y=False,
)

if has_torque:
    fig.add_trace(
        go.Scattergl(x=df["elapsed_s"], y=df["torque_Nm"], name="Torque (Nm)",
                     line=dict(color="#a6e3a1", width=1.2)),
        secondary_y=True,
    )

fig.update_layout(
    title=f"Voltage & Torque — {csv_path.name}",
    template="plotly_dark",
    height=500,
    legend=dict(x=0, y=1.12, orientation="h"),
)
fig.update_xaxes(title_text="Elapsed Time (s)")
fig.update_yaxes(title_text="Voltage (V)", secondary_y=False)
fig.update_yaxes(title_text="Torque (Nm)", secondary_y=True)
fig.show()

## 5. Statistics Summary

In [ ]:
cols = ["voltage_V"]
if has_torque:
    cols.append("torque_Nm")

stats = df[cols].describe()

# Add additional stats
for col in cols:
    stats.loc["range", col] = df[col].max() - df[col].min()
    stats.loc["median", col] = df[col].median()

# Compute actual sample rate from timestamps
dt = df["elapsed_s"].diff().dropna()
stats.loc["actual_rate_Hz", cols[0]] = 1.0 / dt.mean() if dt.mean() > 0 else 0

print(f"Recording: {csv_path.name}")
print(f"Duration:  {df['elapsed_s'].iloc[-1]:.3f} s")
print(f"Samples:   {len(df)}")
print()
stats.round(4)

## 6. Signal Distribution (Histogram)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Voltage Distribution", "Torque Distribution"))

fig.add_trace(
    go.Histogram(x=df["voltage_V"], nbinsx=50, marker_color="#89b4fa", name="Voltage"),
    row=1, col=1,
)

if has_torque:
    fig.add_trace(
        go.Histogram(x=df["torque_Nm"], nbinsx=50, marker_color="#a6e3a1", name="Torque"),
        row=1, col=2,
    )

fig.update_layout(
    template="plotly_dark",
    height=400,
    showlegend=False,
)
fig.update_xaxes(title_text="Voltage (V)", row=1, col=1)
fig.update_xaxes(title_text="Torque (Nm)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.show()

## 7. Compare Multiple Recordings

In [ ]:
# Load all CSV files and overlay them
fig = go.Figure()

colors = px.colors.qualitative.Pastel

for i, path in enumerate(csv_files):
    d = pd.read_csv(path, comment="#")
    if len(d) < 2:
        continue
    fig.add_trace(go.Scattergl(
        x=d["elapsed_s"],
        y=d["voltage_V"],
        mode="lines",
        name=path.name,
        line=dict(color=colors[i % len(colors)], width=1.2),
    ))

fig.update_layout(
    title="All Recordings — Voltage Overlay",
    xaxis_title="Elapsed Time (s)",
    yaxis_title="Voltage (V)",
    template="plotly_dark",
    height=500,
    legend=dict(font=dict(size=10)),
)
fig.show()